# BPM Governance Matrix — LLM-Augmented Decision Support

**Project:** Process Identification in the Endress+Hauser BPM Governance Case  
**Group A5:** Fabian Eppenberger, Ilir Salihi, Korab Hoti, Niels Meyer  
**Course:** Business Process Management (FHNW, FS 2026)

## Purpose

Working prototype of the **LLM-Augmented Decision Support** process (BPMN Model 3 in the report). Two experiments validate the VA/BVA/NVA classification from Chapter 2:

1. **Experiment 1 (NVA, full delegation):** Framework Review — does the LLM replace manual literature review?
2. **Experiment 2 (BVA, partial automation):** RACI Conflict Detection — does the LLM detect planted defects while leaving Accountable review human?

## Architecture

Four-step pipeline mirroring BPMN 3: Receive Request → Collect Data → Analyse → Deliver Package.

## Reproducibility

Each experiment is run `N_RUNS` times (default: 5) and aggregated. Results are reported as mean ± std. This partially addresses the "evaluation scope" limitation noted in §4.5 of the report. Set `ANTHROPIC_API_KEY` in your environment. Model: `claude-sonnet-4-5`.

## 1. Setup and Data Loading

In [14]:
import os
import re
import json
import time
import statistics
import textwrap
from pathlib import Path

import anthropic

# Configuration
MODEL = 'claude-sonnet-4-5'
MAX_TOKENS = 4000
N_RUNS = 5                       # runs per experiment for statistical validation
DATA_DIR = Path('.')

# API pricing (per 1M tokens, public list price for claude-sonnet-4-5)
PRICE_INPUT_PER_MILLION = 3.00
PRICE_OUTPUT_PER_MILLION = 15.00

# Sanity check
if not os.environ.get('ANTHROPIC_API_KEY'):
    raise EnvironmentError(
        'ANTHROPIC_API_KEY not set. Set it via: export ANTHROPIC_API_KEY="sk-..."'
    )

client = anthropic.Anthropic()
print(f'Client ready. Model: {MODEL} | N_RUNS per experiment: {N_RUNS}')

Client ready. Model: claude-sonnet-4-5 | N_RUNS per experiment: 5


In [15]:
# Load the three retrieval sources (workshop log, governance matrix, org structure)
# In a production setting these would be MCP-served from SharePoint, Confluence, BPM tools.

def load_json_safe(path):
    """Load JSON with a clear error if the file is missing or malformed."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f'Required data file missing: {path}')
    except json.JSONDecodeError as e:
        raise ValueError(f'Malformed JSON in {path}: {e}')

workshop_log = load_json_safe(DATA_DIR / 'workshop_log.json')
governance_matrix = load_json_safe(DATA_DIR / 'governance_matrix.json')
org_structure = load_json_safe(DATA_DIR / 'org_structure.json')

# Count items using the actual key names in our data files
n_workshop_items = (
    len(workshop_log.get('agenda_items', []))
    + len(workshop_log.get('open_issues', []))
    + len(workshop_log.get('key_discussion_points', []))
)
n_activities = sum(
    len(activities)
    for activities in governance_matrix.get('activity_groups', {}).values()
)
n_roles = (
    len(org_structure.get('executive_layer', []))
    + len(org_structure.get('functional_units', []))
    + len(org_structure.get('process_organization', []) if isinstance(org_structure.get('process_organization'), list) else [])
)

print(f'workshop_log:      {n_workshop_items} items (agenda + issues + discussion points)')
print(f'governance_matrix: {n_activities} activities across {len(governance_matrix.get("activity_groups", {}))} activity groups')
print(f'org_structure:     {n_roles} roles in executive + functional layers')

workshop_log:      13 items (agenda + issues + discussion points)
governance_matrix: 9 activities across 3 activity groups
org_structure:     13 roles in executive + functional layers


In [16]:
print('workshop_log keys:     ', list(workshop_log.keys()))
print('governance_matrix keys:', list(governance_matrix.keys()))
print('org_structure keys:    ', list(org_structure.keys()))
print()
print('--- workshop_log (first 500 chars) ---')
print(json.dumps(workshop_log, indent=2)[:500])
print()
print('--- governance_matrix (first 500 chars) ---')
print(json.dumps(governance_matrix, indent=2)[:500])
print()
print('--- org_structure (first 500 chars) ---')
print(json.dumps(org_structure, indent=2)[:500])

workshop_log keys:      ['workshop_id', 'date', 'facilitator', 'attendees', 'agenda_items', 'key_discussion_points', 'open_issues', 'decisions_taken']
governance_matrix keys: ['matrix_version', 'last_updated', 'activity_groups']
org_structure keys:     ['company', 'headquarters', 'employees', 'structure_type', 'executive_layer', 'process_organization', 'functional_units', 'bpm_governance_principles']

--- workshop_log (first 500 chars) ---
{
  "workshop_id": "WS-2025-03",
  "date": "2025-03-14",
  "facilitator": "BPM Team",
  "attendees": [
    "COO",
    "Strategic Process Owner - Fulfillment",
    "Strategic Process Owner - Innovation",
    "Head of Quality Management",
    "Head of IT Management",
    "Customer Success Partner"
  ],
  "agenda_items": [
    "Review of current BPM Governance Matrix",
    "Discussion of overlaps between Fulfillment and Innovation",
    "Cross-functional communication in customer experience",
    "

--- governance_matrix (first 500 chars) ---
{
  "matri

## 2. Hardened API Wrapper

The wrapper defends against four common failure modes:

1. **Markdown-fenced JSON** — strips ```` ```json ... ``` ```` wrappers
2. **Malformed JSON** — automatically retries once with stricter instructions
3. **Missing fields** — validates expected schema keys before returning
4. **Missing tokens/timing** — guards against shape differences in the API response

In [17]:
def extract_json(text):
    """Pull a JSON object out of an LLM response, stripping markdown fences if present."""
    text = text.strip()
    # Strip ```json ... ``` or ``` ... ``` fences
    fence_pattern = r'^```(?:json)?\s*(.+?)\s*```$'
    m = re.match(fence_pattern, text, re.DOTALL)
    if m:
        text = m.group(1).strip()
    # Sometimes the model adds preamble before the JSON object; find the first '{'
    if not text.startswith('{') and '{' in text:
        text = text[text.index('{'):]
        last_brace = text.rfind('}')
        if last_brace != -1:
            text = text[:last_brace + 1]
    return text


def call_claude(system_prompt, user_prompt, expected_keys=None, max_retries=1):
    """Call Claude, return parsed JSON plus runtime/token metrics.

    Retries once on JSON parse failure with a stricter system instruction appended.
    Validates that the returned object contains all `expected_keys`.
    """
    last_error = None

    for attempt in range(max_retries + 1):
        effective_system = system_prompt
        if attempt > 0:
            effective_system += (
                '\n\nIMPORTANT: Return ONLY a single valid JSON object. '
                'No prose before or after. No markdown code fences.'
            )

        start = time.time()
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=effective_system,
            messages=[{'role': 'user', 'content': user_prompt}],
        )
        runtime = time.time() - start

        try:
            raw_text = response.content[0].text
        except (AttributeError, IndexError) as e:
            raise RuntimeError(f'Unexpected response shape: {e}')

        input_tokens = getattr(response.usage, 'input_tokens', 0)
        output_tokens = getattr(response.usage, 'output_tokens', 0)

        try:
            parsed = json.loads(extract_json(raw_text))
        except json.JSONDecodeError as e:
            last_error = e
            if attempt < max_retries:
                print(f'    [warn] JSON parse failed (attempt {attempt + 1}), retrying...')
                continue
            raise ValueError(
                f'JSON parse failed after {max_retries + 1} attempts. '
                f'Last error: {e}\nRaw text:\n{raw_text[:500]}'
            )

        if expected_keys:
            missing = [k for k in expected_keys if k not in parsed]
            if missing:
                last_error = f'Missing expected keys: {missing}'
                if attempt < max_retries:
                    print(f'    [warn] Missing keys {missing} (attempt {attempt + 1}), retrying...')
                    continue
                raise ValueError(
                    f'Response missing expected keys after retry: {missing}. '
                    f'Got: {list(parsed.keys())}'
                )

        parsed['_runtime_seconds'] = round(runtime, 2)
        parsed['_input_tokens'] = input_tokens
        parsed['_output_tokens'] = output_tokens
        return parsed

    raise RuntimeError(f'Unreachable; last_error={last_error}')


def compute_cost(input_tokens, output_tokens):
    """Return API cost in USD based on public list pricing."""
    return (
        input_tokens * PRICE_INPUT_PER_MILLION / 1_000_000
        + output_tokens * PRICE_OUTPUT_PER_MILLION / 1_000_000
    )


def summarise(values):
    """Return (mean, stdev) tuple. Stdev is 0 for single-value lists."""
    if len(values) < 2:
        return (values[0] if values else 0.0, 0.0)
    return (statistics.mean(values), statistics.stdev(values))


print('API wrapper ready (hardened: JSON fences, retries, schema validation, defensive metrics)')

API wrapper ready (hardened: JSON fences, retries, schema validation, defensive metrics)


## 3. Output Formatting

Presentation-ready summary blocks. Each experiment ends with one of these — designed so a grader (or anyone watching the demo) can read the result in five seconds.

In [18]:
BOX_WIDTH = 64

def box_header(title):
    print('=' * BOX_WIDTH)
    print(f'  {title}'.ljust(BOX_WIDTH))
    print('=' * BOX_WIDTH)

def box_footer(verdict, passed=True):
    print('-' * BOX_WIDTH)
    marker = 'PASS' if passed else 'FAIL'
    print(f'{marker} -- {verdict}')
    print('=' * BOX_WIDTH)

def box_section(label):
    print(f'\n{label}:')

def box_kv(label, value):
    print(f'  {label:<28s}{value}')

def fmt_mean_std(mean, std, unit='', decimals=2):
    """Format a mean ± std value for table display."""
    if std == 0:
        return f'{mean:.{decimals}f}{unit}'
    return f'{mean:.{decimals}f} +/- {std:.{decimals}f}{unit}'

print('Output formatting helpers ready')

Output formatting helpers ready


## 4. Experiment 1 — Framework Review (NVA, full delegation)

**Objective:** Replace the manual literature review step. Classified as Non-Value Added: no customer value, fully delegable.  
**Manual baseline:** 6--8 hours analyst time.  
**Pass criterion:** Top-ranked framework matches the case-study verdict (Integrated Functional-Department Model) in *all* runs.

In [19]:
system_prompt_exp1 = '''You are a BPM governance analyst supporting Endress+Hauser, 
a 14,000-employee family-owned Swiss measurement and automation company 
transitioning from functional to process-oriented organisation.

Apply the BPM Governance Matrix logic from vom Brocke et al. (2025): 
three activity groups (Strategy, Process, Structure) x four RACI dimensions 
(Responsible, Accountable, Consult, Inform). Accountability stays human. 
You prepare, synthesise, and recommend. You never decide.

Return your output as structured JSON only.'''

user_prompt_exp1 = '''Three governance frameworks are candidates for E+H's BPM redesign:

1. Khusidman's BPM Governance Framework (2010): comprehensive prescriptive framework 
   with detailed tool/method specifications. Top-down reference architecture.

2. Centralised BPM Centre of Excellence (Rosemann, 2015): consolidates BPM expertise 
   into a shared-service unit. Provides standardisation and economies of scale.

3. Integrated Functional-Department Model: BPM tasks embedded into existing functional 
   roles (process owners report to COO; functional roles retain operational control).

Context: E+H has explicitly stated that BPM should NOT be isolated in a separate unit. 
The company values its family-owned culture and the principle of context-awareness. 
Strategic Process Owners already coexist with functional management.

Task: Rank the three frameworks for E+H. Justify each ranking. Provide a decision 
brief for the COO summarising the recommendation and the four validation points the 
BPM Team should confirm before adoption.

Return JSON with this exact schema:
{
  "ranking": [
    {"rank": 1, "framework": "<name>", "justification": "<1-2 sentences>"},
    {"rank": 2, "framework": "<name>", "justification": "<1-2 sentences>"},
    {"rank": 3, "framework": "<name>", "justification": "<1-2 sentences>"}
  ],
  "decision_brief": "<2-3 sentence summary for the COO>",
  "validation_points": ["<point 1>", "<point 2>", "<point 3>", "<point 4>"]
}'''

print(f'Running Experiment 1 (Framework Review) -- {N_RUNS} runs...')
exp1_runs = []
for i in range(N_RUNS):
    print(f'  Run {i + 1}/{N_RUNS}...', end=' ', flush=True)
    result = call_claude(
        system_prompt=system_prompt_exp1,
        user_prompt=user_prompt_exp1,
        expected_keys=['ranking', 'decision_brief', 'validation_points'],
    )
    exp1_runs.append(result)
    print(f'{result["_runtime_seconds"]} s')
print('Done.')

Running Experiment 1 (Framework Review) -- 5 runs...
  Run 1/5... 14.99 s
  Run 2/5... 13.82 s
  Run 3/5... 14.34 s
  Run 4/5... 12.59 s
  Run 5/5... 13.73 s
Done.


In [20]:
CASE_STUDY_TOP_CHOICE = 'Integrated Functional-Department Model'

# Aggregate metrics across runs
runtimes_exp1 = [r['_runtime_seconds'] for r in exp1_runs]
input_tokens_exp1 = [r['_input_tokens'] for r in exp1_runs]
output_tokens_exp1 = [r['_output_tokens'] for r in exp1_runs]
costs_exp1 = [compute_cost(r['_input_tokens'], r['_output_tokens']) for r in exp1_runs]

rt_mean_e1, rt_std_e1 = summarise(runtimes_exp1)
in_mean_e1, _ = summarise(input_tokens_exp1)
out_mean_e1, _ = summarise(output_tokens_exp1)
cost_mean_e1, cost_std_e1 = summarise(costs_exp1)

# Validate every run
matches_exp1 = []
for r in exp1_runs:
    top = r['ranking'][0]['framework']
    ok = ('integrated' in top.lower() or CASE_STUDY_TOP_CHOICE.lower() in top.lower())
    matches_exp1.append(ok)

match_rate_exp1 = sum(matches_exp1) / len(matches_exp1)
passed_exp1 = match_rate_exp1 == 1.0

# Use the first run for the decision brief example
first_run = exp1_runs[0]

box_header('EXPERIMENT 1: FRAMEWORK REVIEW (NVA)')

box_section('Setup')
box_kv('Candidates:', '3 frameworks (Khusidman, CoE, Integrated)')
box_kv('Method:', 'NVA (full delegation to LLM)')
box_kv('Runs:', f'{N_RUNS}')

box_section('Performance (across all runs)')
box_kv('Runtime:', fmt_mean_std(rt_mean_e1, rt_std_e1, ' s'))
box_kv('Tokens in/out (mean):', f'{int(in_mean_e1):,} / {int(out_mean_e1):,}')
box_kv('API cost per run:', fmt_mean_std(cost_mean_e1, cost_std_e1, ' USD', decimals=4))
box_kv('Total cost (5 runs):', f'USD {sum(costs_exp1):.4f}')

box_section('Result (run 1 example)')
for entry in first_run['ranking']:
    framework_short = entry['framework'][:40]
    box_kv(f'Rank {entry["rank"]}:', framework_short)

box_section('Validation')
box_kv('Case verdict:', CASE_STUDY_TOP_CHOICE[:36])
box_kv('Top-rank match rate:', f'{sum(matches_exp1)}/{N_RUNS} runs ({match_rate_exp1 * 100:.0f}%)')
box_kv('Human review points:', f'{len(first_run["validation_points"])} listed (run 1)')

box_footer(
    f'Top rank matches case-study verdict in {sum(matches_exp1)}/{N_RUNS} runs; '
    'human-review boundary preserved.'
    if passed_exp1 else f'Match rate = {match_rate_exp1 * 100:.0f}%. Inconsistent ranking.',
    passed=passed_exp1,
)

# Wrapped long-text sections for clean screenshot capture
WRAP_WIDTH = 64

print('\nDecision brief for COO (run 1):')
print(textwrap.fill(
    first_run['decision_brief'],
    width=WRAP_WIDTH,
    initial_indent='  ',
    subsequent_indent='  ',
))

print('\nHuman validation points (run 1):')
for i, point in enumerate(first_run['validation_points'], 1):
    print(textwrap.fill(
        point,
        width=WRAP_WIDTH,
        initial_indent=f'  {i}. ',
        subsequent_indent='     ',
    ))

  EXPERIMENT 1: FRAMEWORK REVIEW (NVA)                          

Setup:
  Candidates:                 3 frameworks (Khusidman, CoE, Integrated)
  Method:                     NVA (full delegation to LLM)
  Runs:                       5

Performance (across all runs):
  Runtime:                    13.89 +/- 0.89 s
  Tokens in/out (mean):       537 / 546
  API cost per run:           0.0098 +/- 0.0004 USD
  Total cost (5 runs):        USD 0.0490

Result (run 1 example):
  Rank 1:                     Integrated Functional-Department Model
  Rank 2:                     Khusidman's BPM Governance Framework (20
  Rank 3:                     Centralised BPM Centre of Excellence (Ro

Validation:
  Case verdict:               Integrated Functional-Department Mod
  Top-rank match rate:        5/5 runs (100%)
  Human review points:        4 listed (run 1)
----------------------------------------------------------------
PASS -- Top rank matches case-study verdict in 5/5 runs; human-review boundary

## 5. Experiment 2 — RACI Conflict Detection (BVA, partial automation)

**Objective:** Detect overlaps and blind spots in a deliberately flawed governance matrix.  
**Manual baseline:** 2--3 hours analyst time; humans typically miss one defect per audit.  
**Pass criterion:** Recall = 4/4 on planted defects across runs (consistency check).

In [21]:
system_prompt_exp2 = '''You are a BPM governance analyst auditing a RACI matrix 
at Endress+Hauser. Apply RACI rules strictly:

1. Each activity must have exactly one Accountable (one neck to wring).
2. Each activity must have at least one Responsible.
3. Consult roles must have relevant expertise.
4. Cross-functional activities must Inform impacted stakeholders.

Apply BPM governance principles from vom Brocke et al. (2025): 
Strategy/Process/Structure x RACI logic. Accountability stays human. 
You prepare, synthesise, and recommend. You never decide.

Cross-reference findings with the workshop log to identify any latent 
governance debt already known but not yet resolved.

Return your output as structured JSON only.'''

user_prompt_exp2 = f'''Audit the following governance matrix and identify ALL defects.

GOVERNANCE MATRIX:
{json.dumps(governance_matrix, indent=2)}

ORG STRUCTURE:
{json.dumps(org_structure, indent=2)}

WORKSHOP LOG:
{json.dumps(workshop_log, indent=2)}

Task: List every RACI defect found. Categorise by type. Indicate severity 
(High/Medium/Low) and propose a fix.

Return JSON with this exact schema:
{{
  "defects": [
    {{
      "activity": "<activity name>",
      "type": "<one of: duplicate_accountable | missing_responsible | misplaced_consult | missing_inform | other>",
      "severity": "<High | Medium | Low>",
      "description": "<1 sentence>",
      "suggested_fix": "<1 sentence>",
      "workshop_log_corroboration": "<quote or null>"
    }}
  ]
}}'''

print(f'Running Experiment 2 (RACI Conflict Detection) -- {N_RUNS} runs...')
exp2_runs = []
for i in range(N_RUNS):
    print(f'  Run {i + 1}/{N_RUNS}...', end=' ', flush=True)
    result = call_claude(
        system_prompt=system_prompt_exp2,
        user_prompt=user_prompt_exp2,
        expected_keys=['defects'],
    )
    exp2_runs.append(result)
    print(f'{result["_runtime_seconds"]} s -- {len(result["defects"])} defects')
print('Done.')

Running Experiment 2 (RACI Conflict Detection) -- 5 runs...
  Run 1/5... 22.18 s -- 11 defects
  Run 2/5... 21.9 s -- 11 defects
  Run 3/5... 23.41 s -- 12 defects
  Run 4/5... 19.08 s -- 10 defects
  Run 5/5... 23.89 s -- 12 defects
Done.


In [22]:
PLANTED_DEFECTS = [
    'duplicate_accountable',
    'missing_responsible',
    'misplaced_consult',
    'missing_inform',
]

# Per-run validation
recall_per_run = []
planted_found_per_run = []
total_defects_per_run = []
bonus_per_run = []
corroborated_per_run = []
missed_planted_overall = set()

for r in exp2_runs:
    detected_types = {d.get('type', 'other').lower() for d in r['defects']}
    matched = [p for p in PLANTED_DEFECTS if p in detected_types]
    missed = [p for p in PLANTED_DEFECTS if p not in detected_types]
    missed_planted_overall.update(missed)
    
    recall_per_run.append(len(matched) / len(PLANTED_DEFECTS))
    planted_found_per_run.append(len(matched))
    total_defects_per_run.append(len(r['defects']))
    bonus_per_run.append(len(r['defects']) - len(matched))
    corroborated_per_run.append(sum(
        1 for d in r['defects']
        if d.get('workshop_log_corroboration')
        and d['workshop_log_corroboration'] not in (None, 'null', '')
    ))

# Aggregate metrics
runtimes_exp2 = [r['_runtime_seconds'] for r in exp2_runs]
input_tokens_exp2 = [r['_input_tokens'] for r in exp2_runs]
output_tokens_exp2 = [r['_output_tokens'] for r in exp2_runs]
costs_exp2 = [compute_cost(r['_input_tokens'], r['_output_tokens']) for r in exp2_runs]

rt_mean_e2, rt_std_e2 = summarise(runtimes_exp2)
in_mean_e2, _ = summarise(input_tokens_exp2)
out_mean_e2, _ = summarise(output_tokens_exp2)
cost_mean_e2, cost_std_e2 = summarise(costs_exp2)
recall_mean, recall_std = summarise(recall_per_run)
total_mean, total_std = summarise(total_defects_per_run)
bonus_mean, bonus_std = summarise(bonus_per_run)
corr_mean, corr_std = summarise(corroborated_per_run)

runs_with_full_recall = sum(1 for r in recall_per_run if r == 1.0)
passed_exp2 = runs_with_full_recall == N_RUNS

# Detection rate per planted defect (across runs)
per_defect_hits = {p: 0 for p in PLANTED_DEFECTS}
for r in exp2_runs:
    detected_types = {d.get('type', 'other').lower() for d in r['defects']}
    for p in PLANTED_DEFECTS:
        if p in detected_types:
            per_defect_hits[p] += 1

box_header('EXPERIMENT 2: RACI CONFLICT DETECTION (BVA)')

box_section('Setup')
box_kv('Planted defects:', f'{len(PLANTED_DEFECTS)}')
box_kv('Method:', 'BVA (partial automation, human reviews)')
box_kv('Runs:', f'{N_RUNS}')

box_section('Performance (across all runs)')
box_kv('Runtime:', fmt_mean_std(rt_mean_e2, rt_std_e2, ' s'))
box_kv('Tokens in/out (mean):', f'{int(in_mean_e2):,} / {int(out_mean_e2):,}')
box_kv('API cost per run:', fmt_mean_std(cost_mean_e2, cost_std_e2, ' USD', decimals=4))
box_kv('Total cost (5 runs):', f'USD {sum(costs_exp2):.4f}')

box_section('Results (across all runs)')
box_kv('Defects flagged (mean):', fmt_mean_std(total_mean, total_std, '', decimals=1))
box_kv('Planted recall:', f'{recall_mean * 100:.0f}% (+/- {recall_std * 100:.0f}%)')
box_kv('Bonus findings (mean):', fmt_mean_std(bonus_mean, bonus_std, '', decimals=1))
box_kv('Workshop-log links:', fmt_mean_std(corr_mean, corr_std, '', decimals=1))
box_kv('Runs with full recall:', f'{runs_with_full_recall}/{N_RUNS}')

box_section('Per-defect detection rate')
for p, hits in per_defect_hits.items():
    status = '[OK ]' if hits == N_RUNS else '[----]' if hits == 0 else '[~~]'
    print(f'  {status} {p:<26s} {hits}/{N_RUNS} runs')

box_footer(
    f'Full recall in {runs_with_full_recall}/{N_RUNS} runs; consistent detection. BVA criterion met.'
    if passed_exp2 else f'Full recall in only {runs_with_full_recall}/{N_RUNS} runs. '
    f'Sometimes missed: {missed_planted_overall}',
    passed=passed_exp2,
)

  EXPERIMENT 2: RACI CONFLICT DETECTION (BVA)                   

Setup:
  Planted defects:            4
  Method:                     BVA (partial automation, human reviews)
  Runs:                       5

Performance (across all runs):
  Runtime:                    22.09 +/- 1.88 s
  Tokens in/out (mean):       1,830 / 1,358
  API cost per run:           0.0259 +/- 0.0015 USD
  Total cost (5 runs):        USD 0.1293

Results (across all runs):
  Defects flagged (mean):     11.2 +/- 0.8
  Planted recall:             100% (+/- 0%)
  Bonus findings (mean):      7.2 +/- 0.8
  Workshop-log links:         7.0 +/- 1.0
  Runs with full recall:      5/5

Per-defect detection rate:
  [OK ] duplicate_accountable      5/5 runs
  [OK ] missing_responsible        5/5 runs
  [OK ] misplaced_consult          5/5 runs
  [OK ] missing_inform             5/5 runs
----------------------------------------------------------------
PASS -- Full recall in 5/5 runs; consistent detection. BVA criterion met.


## 6. Cost-Benefit Summary

Total economic analysis across both experiments. Measured values (mean of 5 runs each) feed Table 4.2 in the report.

In [23]:
rows = [
    ('Framework review (NVA)',  '6--8 h',  f'{rt_mean_e1:.1f} s',  cost_mean_e1),
    ('RACI matrix audit (BVA)', '2--3 h',  f'{rt_mean_e2:.1f} s',  cost_mean_e2),
]
total_cost_per_run = cost_mean_e1 + cost_mean_e2
total_cost_all_runs = sum(costs_exp1) + sum(costs_exp2)

box_header('COST-BENEFIT SUMMARY (MEAN OF 5 RUNS EACH)')

print()
print(f'  {"Activity":<26s} {"Manual":<10s} {"LLM (mean)":<12s} {"API cost":>12s}')
print('  ' + '-' * (BOX_WIDTH - 4))
for activity, manual, llm, cost in rows:
    print(f'  {activity:<26s} {manual:<10s} {llm:<12s} USD {cost:>7.4f}')
print('  ' + '-' * (BOX_WIDTH - 4))
total_runtime_str = f'~{rt_mean_e1 + rt_mean_e2:.0f} s'
print(f'  {"Total (per audit cycle)":<26s} {"8--11 h":<10s} {total_runtime_str:<12s} USD {total_cost_per_run:>7.4f}')

print()
print(f'  Per-audit cost (mean):    USD {total_cost_per_run:.4f}')
print(f'  Total spent (10 runs):    USD {total_cost_all_runs:.4f}')
print(f'  Manual baseline:          8--11 hours analyst time')
print(f'  Conclusion: continuous matrix maintenance is economically trivial')
print(f'              compared to consulting time it replaces.')

print('=' * BOX_WIDTH)

  COST-BENEFIT SUMMARY (MEAN OF 5 RUNS EACH)                    

  Activity                   Manual     LLM (mean)       API cost
  ------------------------------------------------------------
  Framework review (NVA)     6--8 h     13.9 s       USD  0.0098
  RACI matrix audit (BVA)    2--3 h     22.1 s       USD  0.0259
  ------------------------------------------------------------
  Total (per audit cycle)    8--11 h    ~36 s        USD  0.0357

  Per-audit cost (mean):    USD 0.0357
  Total spent (10 runs):    USD 0.1783
  Manual baseline:          8--11 hours analyst time
  Conclusion: continuous matrix maintenance is economically trivial
              compared to consulting time it replaces.


## 7. Conclusion

Both experiments confirm the central design claim of the Process Analysis chapter: **VA/BVA/NVA classification is a workable criterion for determining LLM automation depth.**

- **NVA step** (framework review): fully delegated. Output matches the case-study verdict consistently across runs in seconds rather than hours.
- **BVA step** (matrix audit): detects all planted defects, surfaces latent governance debt from the workshop log, and preserves the human *Accountable* review step.

API cost remains below USD 0.10 per request, supporting the cost-benefit case for continuous matrix maintenance — the "living document" the original case study called for (vom Brocke et al., 2025, p. 235).